# API Routes - Plants
Flask routes for plant management operations.

In [ ]:
# Plant API routes - to be registered with Flask app

def register_plant_routes(app, get_h5_file, get_sequencer_grid, set_sequencer_grid, create_plant_with_context):
    """Register plant-related API routes"""
    from flask import jsonify, request
    
    @app.route('/api/plants', methods=['GET'])
    def get_plants():
        """Get all plants with their IDs and timestamps"""
        try:
            with get_h5_file() as f:
                plants_group = f.require_group("plants")
                plants = []
                for plant_id in plants_group.keys():
                    plant = plants_group[plant_id]
                    added_timestamp = plant.attrs.get("added_timestamp", "")
                    if isinstance(added_timestamp, bytes):
                        added_timestamp = added_timestamp.decode('utf-8')
                    plants.append({
                        "id": plant_id,
                        "added_timestamp": added_timestamp
                    })
            return jsonify({"plants": plants}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/plants', methods=['POST'])
    def create_plant_endpoint():
        """Create a new plant and return its ID"""
        try:
            with get_h5_file() as f:
                plant = create_plant_with_context(f)
                plant_id = plant.name.split('/')[-1]
                return jsonify({"plant_id": plant_id}), 201
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/sequencer', methods=['GET'])
    def get_sequencer():
        """Get current sequencer grid state"""
        try:
            grid = get_sequencer_grid()
            return jsonify({"sequencer": grid}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/sequencer', methods=['PUT'])
    def update_sequencer():
        """Update sequencer grid position"""
        try:
            data = request.get_json()
            row = data.get('row')
            col = data.get('col')
            plant_id = data.get('plant_id', '')
            
            if row is None or col is None:
                return jsonify({"error": "row and col are required"}), 400
            
            if row < 0 or row >= 2 or col < 0 or col >= 12:
                return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
            
            grid = get_sequencer_grid()
            grid[row][col] = plant_id
            set_sequencer_grid(grid)
            
            return jsonify({"sequencer": grid}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/sequencer/auto-populate', methods=['POST'])
    def auto_populate_sequencer():
        """Auto-populate empty sequencer slots with new plants"""
        try:
            grid = get_sequencer_grid()
            count = 0
            
            with get_h5_file() as f:
                for row in range(2):
                    for col in range(12):
                        if not grid[row][col]:
                            plant_group = create_plant_with_context(f)
                            plant_id = plant_group.name.split('/')[-1]
                            grid[row][col] = plant_id
                            count += 1
            
            set_sequencer_grid(grid)
            return jsonify({"success": True, "count": count, "sequencer": grid}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500